# SpiderFoot API Key Hunter

Automated API key acquisition system for SpiderFoot modules.

**Account:** `api-manager@blking.net`  
**Temp Email:** `agogfze@mailto.plus` → check at https://tempmail.plus/

## Sections
1. Imports & Setup
2. Service Definitions (working set)
3. Database Layer
4. Status & Reporting
5. Notification Utilities
6. Initialize Database
7. Manual Workflow
8. View Current Progress
9. Export Keys
10. Full 116-Service Reference
11. SpiderFoot Configuration Methods
12. SpiderFoot DB Utilities
13. Troubleshooting

## 1. Imports & Setup

In [ ]:
import json
import sqlite3
import re
import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict

DB_PATH = 'api_keys_progress.db'
SPIDERFOOT_DB = '/stuff/spiderfoot/spiderfoot.db'  # SpiderFoot's main database
print(f'Hunter DB: {DB_PATH}')
print(f'SpiderFoot DB: {SPIDERFOOT_DB}')

## 2. Service Definitions (working set)

High-priority services being actively acquired.

In [ ]:
# Working set - services actively being acquired
SERVICES = {
    # Security & Threat Intelligence - FREE
    'AlienVault OTX': {
        'url': 'https://otx.alienvault.com/#signup',
        'tier': 'Free', 'priority': 'High', 'category': 'Security',
        'status': 'in_progress',
        'notes': 'Form filled, needs CAPTCHA solve, then check email agogfze@mailto.plus'
    },
    'AbuseIPDB': {
        'url': 'https://www.abuseipdb.com/register',
        'tier': 'Free', 'priority': 'High', 'category': 'Security'
    },
    'GreyNoise Community': {
        'url': 'https://www.greynoise.io/viz/account/api-key',
        'tier': 'Free', 'priority': 'High', 'category': 'Security',
        'notes': 'Free community API key after signup'
    },
    'GreyNoise': {
        'url': 'https://www.greynoise.io/viz/signup',
        'tier': 'Freemium', 'priority': 'High', 'category': 'Security'
    },
    'Hybrid Analysis': {
        'url': 'https://www.hybrid-analysis.com/signup',
        'tier': 'Free', 'priority': 'Medium', 'category': 'Security'
    },
    'Pulsedive': {
        'url': 'https://pulsedive.com/register',
        'tier': 'Free', 'priority': 'Medium', 'category': 'Security',
        'notes': 'Email signup, API key in profile'
    },
    'SHODAN': {
        'url': 'https://account.shodan.io/register',
        'tier': 'Paid', 'priority': 'High', 'category': 'Security'
    },
    'VirusTotal': {
        'url': 'https://www.virustotal.com/gui/join-us',
        'login_url': 'https://www.virustotal.com/gui/sso/google',
        'tier': 'Freemium', 'priority': 'High', 'category': 'Security',
        'notes': 'Sign in with Google SSO, then navigate to API key page'
    },
    'BinaryEdge': {
        'url': 'https://app.binaryedge.io/sign-up',
        'tier': 'Freemium', 'priority': 'Medium', 'category': 'Search'
    },
    'Censys': {
        'url': 'https://censys.io/register',
        'tier': 'Freemium', 'priority': 'High', 'category': 'Search',
        'notes': '50 queries/month free'
    },
    'Google Safe Browsing': {
        'url': 'https://console.cloud.google.com/',
        'tier': 'Free', 'priority': 'Medium', 'category': 'Search',
        'notes': 'Requires Google account, enable Safe Browsing API'
    },
    'Project Discovery': {
        'url': 'https://cloud.projectdiscovery.io/',
        'tier': 'Freemium', 'priority': 'Medium', 'category': 'Search'
    },
    'ZoomEye': {
        'url': 'https://www.zoomeye.org/register',
        'tier': 'Paid', 'priority': 'Medium', 'category': 'Search'
    },
    'EmailRep': {
        'url': 'https://emailrep.io/key',
        'tier': 'Free', 'priority': 'Medium', 'category': 'Email',
        'notes': 'Just enter email, key sent immediately'
    },
    'HaveIBeenPwned': {
        'url': 'https://haveibeenpwned.com/API/Key',
        'tier': 'Free', 'priority': 'High', 'category': 'Email',
        'notes': 'Requires verification, $3.50/month minimum'
    },
    'Hunter.io': {
        'url': 'https://hunter.io/users/sign_up',
        'tier': 'Freemium', 'priority': 'High', 'category': 'Email',
        'notes': 'Email signup, API key in account settings'
    },
    'LeakIX': {
        'url': 'https://leakix.net/',
        'tier': 'Free', 'priority': 'Medium', 'category': 'Email',
        'notes': 'Free API key after account creation'
    },
    'CertSpotter': {
        'url': 'https://sslmate.com/certspotter/pricing',
        'tier': 'Free', 'priority': 'High', 'category': 'Domain',
        'notes': 'Free tier available, API key after signup'
    },
    'SecurityTrails': {
        'url': 'https://securitytrails.com/app/signup',
        'tier': 'Paid', 'priority': 'High', 'category': 'Domain'
    },
    'ViewDNS.info': {
        'url': 'https://viewdns.info/api/',
        'tier': 'Freemium', 'priority': 'Medium', 'category': 'Domain'
    },
    'CriminalIP': {
        'url': 'https://www.criminalip.io/',
        'tier': 'Freemium', 'priority': 'High', 'category': 'IP'
    },
    'IPInfo.io': {
        'url': 'https://ipinfo.io/signup',
        'tier': 'Freemium', 'priority': 'High', 'category': 'IP',
        'notes': '50k requests/month free'
    },
    'IPQualityScore': {
        'url': 'https://www.ipqualityscore.com/create-account',
        'tier': 'Freemium', 'priority': 'Medium', 'category': 'IP'
    },
    'Etherscan': {
        'url': 'https://etherscan.io/register',
        'tier': 'Free', 'priority': 'Medium', 'category': 'Blockchain',
        'notes': 'Email signup, API key in My API-KEYs section'
    },
    'FullHunt': {
        'url': 'https://fullhunt.io/',
        'tier': 'Paid', 'priority': 'Medium', 'category': 'Business'
    },
    'Onyphe': {
        'url': 'https://www.onyphe.io/',
        'tier': 'Freemium', 'priority': 'Medium', 'category': 'Business'
    },
    'IntelligenceX': {
        'url': 'https://intelx.io/signup',
        'tier': 'Paid', 'priority': 'High', 'category': 'OSINT'
    },
    'PasteBin': {
        'url': 'https://pastebin.com/doc_scraping_api',
        'tier': 'Freemium', 'priority': 'Medium', 'category': 'OSINT'
    },
    'GitHub': {
        'url': 'https://github.com/settings/tokens',
        'tier': 'Free', 'priority': 'High', 'category': 'Dev',
        'notes': 'Create personal access token'
    },
}

FREE_HIGH     = [(k, v) for k, v in SERVICES.items() if v.get('tier') == 'Free'     and v.get('priority') == 'High']
FREEMIUM_HIGH = [(k, v) for k, v in SERVICES.items() if v.get('tier') == 'Freemium' and v.get('priority') == 'High']

print(f'Working set: {len(SERVICES)} services')
print(f'Free + High: {len(FREE_HIGH)}  |  Freemium + High: {len(FREEMIUM_HIGH)}')

## 3. Database Layer

In [ ]:
class APIKeyDatabase:
    """SQLite database for tracking API key acquisition progress"""

    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('''
                CREATE TABLE IF NOT EXISTS api_keys (
                    service    TEXT PRIMARY KEY,
                    url        TEXT,
                    tier       TEXT,
                    priority   TEXT,
                    category   TEXT,
                    api_key    TEXT,
                    status     TEXT DEFAULT 'pending',
                    notes      TEXT,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    updated_at TIMESTAMP
                )
            ''')
            conn.execute('''
                CREATE TABLE IF NOT EXISTS credentials (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    service    TEXT,
                    username   TEXT,
                    email      TEXT,
                    password   TEXT,
                    temp_email TEXT,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            ''')
            conn.execute('''
                CREATE TABLE IF NOT EXISTS signup_attempts (
                    id            INTEGER PRIMARY KEY AUTOINCREMENT,
                    service       TEXT,
                    timestamp     TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    success       BOOLEAN,
                    error_message TEXT
                )
            ''')

    def upsert_service(self, service: str, data: dict):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('''
                INSERT INTO api_keys (service, url, tier, priority, category, status, notes, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(service) DO UPDATE SET
                    url=excluded.url, tier=excluded.tier, priority=excluded.priority,
                    category=excluded.category, notes=excluded.notes, updated_at=excluded.updated_at
            ''', (
                service, data.get('url'), data.get('tier'), data.get('priority'),
                data.get('category'), data.get('status', 'pending'),
                data.get('notes', ''), datetime.now()
            ))

    def save_api_key(self, service: str, api_key: str):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "UPDATE api_keys SET api_key=?, status='completed', updated_at=? WHERE service=?",
                (api_key, datetime.now(), service)
            )
        print(f'Saved API key for {service}')

    def log_attempt(self, service: str, success: bool, error_msg: str = None):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                'INSERT INTO signup_attempts (service, success, error_message) VALUES (?, ?, ?)',
                (service, success, error_msg)
            )

    def save_credentials(self, service: str, username: str, email: str, password: str, temp_email: str = None):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                'INSERT INTO credentials (service, username, email, password, temp_email) VALUES (?, ?, ?, ?, ?)',
                (service, username, email, password, temp_email)
            )

    def get_all_keys(self) -> list:
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute(
                'SELECT service, api_key, status, tier, priority FROM api_keys ORDER BY priority, tier, service'
            ).fetchall()

    def get_completed(self) -> list:
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute(
                "SELECT service, api_key FROM api_keys WHERE api_key IS NOT NULL AND status='completed'"
            ).fetchall()

    def get_summary(self) -> dict:
        with sqlite3.connect(self.db_path) as conn:
            total     = conn.execute('SELECT COUNT(*) FROM api_keys').fetchone()[0]
            completed = conn.execute("SELECT COUNT(*) FROM api_keys WHERE status='completed'").fetchone()[0]
            in_prog   = conn.execute("SELECT COUNT(*) FROM api_keys WHERE status='in_progress'").fetchone()[0]
            pending   = conn.execute("SELECT COUNT(*) FROM api_keys WHERE status='pending'").fetchone()[0]
        return {'total': total, 'completed': completed, 'in_progress': in_prog, 'pending': pending}

    def export_env(self) -> str:
        lines = []
        for service, api_key in self.get_completed():
            env_name = service.upper().replace(' ', '_').replace('.', '_')
            lines.append(f'{env_name}_API_KEY={api_key}')
        return '\n'.join(lines)

    def export_json(self) -> str:
        rows = self.get_all_keys()
        data = [{'service': r[0], 'api_key': r[1], 'status': r[2], 'tier': r[3], 'priority': r[4]} for r in rows]
        return json.dumps(data, indent=2)


print('APIKeyDatabase defined')

## 4. Status & Reporting

In [ ]:
def status_report(db: APIKeyDatabase = None):
    if db is None:
        db = APIKeyDatabase()
    s = db.get_summary()
    total = s['total'] or 1

    print('=' * 60)
    print('API KEY ACQUISITION STATUS')
    print('=' * 60)
    print(f"Total    : {s['total']}")
    print(f"Done     : {s['completed']}  ({s['completed']/total*100:.1f}%)")
    print(f"Progress : {s['in_progress']}")
    print(f"Pending  : {s['pending']}")
    print('=' * 60)

    with sqlite3.connect(db.db_path) as conn:
        for priority in ['High', 'Medium', 'Low']:
            row = conn.execute(
                "SELECT COUNT(*), SUM(status='completed') FROM api_keys WHERE priority=?", (priority,)
            ).fetchone()
            if row[0]:
                print(f'{priority:6} priority: {row[1] or 0}/{row[0]}')
        print('-' * 60)
        for tier in ['Free', 'Freemium', 'Paid']:
            row = conn.execute(
                "SELECT COUNT(*), SUM(status='completed') FROM api_keys WHERE tier=?", (tier,)
            ).fetchone()
            if row[0]:
                print(f'{tier:8} tier: {row[1] or 0}/{row[0]}')


print('status_report() defined')

## 5. Notification Utilities

In [ ]:
def notify(message: str):
    """Print a loud attention banner and ring the terminal bell"""
    print('\n' * 2)
    print('\033[91m' + '#' * 60 + '\033[0m')
    print('\033[91m#  ATTENTION NEEDED' + ' ' * 40 + '#\033[0m')
    print(f'\033[93m#  {message:<56}#\033[0m')
    print('\033[91m' + '#' * 60 + '\033[0m')
    print()
    for _ in range(3):
        sys.stdout.write('\a')
        sys.stdout.flush()
    try:
        subprocess.run(['paplay', '/usr/share/sounds/freedesktop/stereo/bell.oga'],
                       capture_output=True, timeout=1)
    except Exception:
        pass


print('notify() defined')

## 6. Initialize Database

Run once to seed all services.

In [ ]:
db = APIKeyDatabase()

for service_name, service_data in SERVICES.items():
    db.upsert_service(service_name, service_data)

db.save_credentials(
    service='AlienVault OTX',
    username='spiderfoot_hunter',
    email='agogfze@mailto.plus',
    password='SpiderFoot2024!Secure',
    temp_email='agogfze@mailto.plus'
)

print(f'Initialized {len(SERVICES)} services into {DB_PATH}')
status_report(db)

## 7. Manual Workflow

Walk through services one by one. For each:
1. Instructions print
2. Paste the API key (or press Enter to skip)
3. Key saved to database

### 7a. Free + High Priority (start here)

In [ ]:
def collect_keys(service_list: list, db: APIKeyDatabase):
    for name, data in service_list:
        print(f"\n{'='*60}")
        print(f"Service : {name}")
        print(f"URL     : {data.get('url')}")
        print(f"Tier    : {data.get('tier')}  |  Priority: {data.get('priority')}")
        if data.get('notes'):
            print(f"Notes   : {data['notes']}")
        print(f"{'='*60}")
        notify(f"Open {name} and get the API key")

        raw = input(f'Paste API key for {name} (Enter=skip): ').strip()
        if raw and raw.lower() != 'skip':
            db.save_api_key(name, raw)
            db.log_attempt(name, True)
        else:
            print(f'Skipped {name}')
            db.log_attempt(name, False, 'skipped')

    print('\nBatch done.')
    status_report(db)


collect_keys(FREE_HIGH, db)

### 7b. Freemium + High Priority

In [ ]:
collect_keys(FREEMIUM_HIGH, db)

### 7c. Save a single key manually

In [ ]:
SERVICE_NAME = 'VirusTotal'
API_KEY      = 'PASTE_KEY_HERE'

if API_KEY != 'PASTE_KEY_HERE':
    db.save_api_key(SERVICE_NAME, API_KEY)
else:
    print('Set API_KEY above first')

## 8. View Current Progress

In [ ]:
status_report(db)

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute(
        'SELECT service, tier, priority, status, SUBSTR(api_key,1,12) FROM api_keys ORDER BY priority, tier'
    ).fetchall()

print(f"{'Service':<25} {'Tier':<10} {'Priority':<8} {'Status':<12} Key preview")
print('-' * 75)
for r in rows:
    key_preview = (r[4] + '...') if r[4] else ''
    print(f'{r[0]:<25} {r[1]:<10} {r[2]:<8} {r[3]:<12} {key_preview}')

## 9. Export Keys

In [ ]:
# Module → SpiderFoot config key mapping
SPIDERFOOT_KEY_MAP = {
    'AbuseIPDB':           'sfp_abuseipdb:api_key',
    'AlienVault OTX':      'sfp_alienvault:api_key',
    'BinaryEdge':          'sfp_binaryedge:binaryedge_api_key',
    'Censys':              'sfp_censys:censys_api_key_uid',
    'CertSpotter':         'sfp_certspotter:api_key',
    'CriminalIP':          'sfp_criminalip:api_key',
    'EmailRep':            'sfp_emailrep:api_key',
    'Etherscan':           'sfp_etherscan:api_key',
    'FullHunt':            'sfp_fullhunt:api_key',
    'Google Safe Browsing':'sfp_googlesafebrowsing:api_key',
    'GreyNoise':           'sfp_greynoise:api_key',
    'GreyNoise Community': 'sfp_greynoise_community:api_key',
    'HaveIBeenPwned':      'sfp_haveibeenpwned:api_key',
    'Hunter.io':           'sfp_hunter:api_key',
    'Hybrid Analysis':     'sfp_hybrid_analysis:api_key',
    'IntelligenceX':       'sfp_intelx:api_key',
    'IPInfo.io':           'sfp_ipinfo:api_key',
    'IPQualityScore':      'sfp_ipqualityscore:api_key',
    'LeakIX':              'sfp_leakix:api_key',
    'Onyphe':              'sfp_onyphe:api_key',
    'PasteBin':            'sfp_pastebin:api_key',
    'Pulsedive':           'sfp_pulsedive:api_key',
    'SecurityTrails':      'sfp_securitytrails:api_key',
    'SHODAN':              'sfp_shodan:api_key',
    'ViewDNS.info':        'sfp_viewdns:api_key',
    'VirusTotal':          'sfp_virustotal:api_key',
}

print('.env format:')
print(db.export_env() or '(none yet)')
print()
print('-- SpiderFoot SQL:')
for service, api_key in db.get_completed():
    config_key = SPIDERFOOT_KEY_MAP.get(service, f'sfp_unknown:{service.lower()}')
    print(f"INSERT INTO tbl_config (config_name, config_value) VALUES ('{config_key}', '{api_key}');")

In [ ]:
# Write api_keys.env to disk
env_content = db.export_env()
if env_content:
    Path('api_keys.env').write_text(env_content)
    print(f'Written to api_keys.env ({len(env_content)} bytes)')
else:
    print('No completed keys yet')

---
## 10. Full 116-Service Reference

*From `documentation/SPIDERFOOT_API_KEYS.md` — last updated 2025-12-18*

### Executive Summary

| Metric | Value |
|--------|-------|
| **Total Modules** | 278 |
| **Modules Requiring API Keys** | 116 (41.7%) |
| **Categories** | 10 |
| **Free Tier Services** | ~45 |
| **Paid Tier Services** | ~60 |
| **Freemium Services** | ~11 |

### Top 10 Priority Keys (Configure First)

1. **VirusTotal** — Malware detection & file reputation
2. **Shodan** — Internet search & device enumeration
3. **HaveIBeenPwned** — Breach database searches
4. **SecurityTrails** — WHOIS & DNS history
5. **Hunter.io** — Email discovery
6. **Censys** — Internet scanning & certificate data
7. **AlienVault OTX** — Threat intelligence feeds
8. **CertSpotter** — SSL certificate monitoring
9. **Cisco Umbrella** — Malware & phishing detection
10. **IPInfo.io** — IP geolocation & metadata

In [ ]:
# Complete 116-service catalog with module and config key info
ALL_116_SERVICES = [
    # (id, service, module, config_key, tier, priority, category)

    # 1. Security & Threat Intelligence
    (1,  'AbuseIPDB',              'sfp_abuseipdb',          'api_key',                                          'Free',     'High',   'Security'),
    (2,  'AlienVault OTX',         'sfp_alienvault',          'api_key',                                          'Free',     'High',   'Security'),
    (3,  'GreyNoise',              'sfp_greynoise',           'api_key',                                          'Freemium', 'High',   'Security'),
    (4,  'GreyNoise Community',    'sfp_greynoise_community', 'api_key',                                          'Free',     'High',   'Security'),
    (5,  'Hybrid Analysis',        'sfp_hybrid_analysis',     'api_key',                                          'Free',     'Medium', 'Security'),
    (6,  'Mandiant Threat Intel',  'sfp_mandiant_ti',         'api_key',                                          'Paid',     'Medium', 'Security'),
    (7,  'Recorded Future',        'sfp_recordedfuture',      'api_key',                                          'Paid',     'High',   'Security'),
    (8,  'SHODAN',                 'sfp_shodan',              'api_key',                                          'Paid',     'High',   'Security'),
    (9,  'VirusTotal',             'sfp_virustotal',          'api_key',                                          'Freemium', 'High',   'Security'),
    (10, 'Pulsedive',              'sfp_pulsedive',           'api_key',                                          'Free',     'Medium', 'Security'),

    # 2. Search & Discovery
    (11, 'BinaryEdge',             'sfp_binaryedge',          'binaryedge_api_key',                               'Freemium', 'Medium', 'Search'),
    (12, 'Bing Search',            'sfp_bingsearch',          'api_key',                                          'Paid',     'Medium', 'Search'),
    (13, 'Bing Shared IPs',        'sfp_bingsharedip',        'api_key',                                          'Paid',     'Low',    'Search'),
    (14, 'Censys',                 'sfp_censys',              'censys_api_key_uid,censys_api_key_secret',          'Freemium', 'High',   'Search'),
    (15, 'Google Search',          'sfp_googlesearch',        'api_key',                                          'Paid',     'High',   'Search'),
    (16, 'Google Maps',            'sfp_googlemaps',          'api_key',                                          'Paid',     'Low',    'Search'),
    (17, 'Google Safe Browsing',   'sfp_googlesafebrowsing',  'api_key',                                          'Free',     'Medium', 'Search'),
    (18, 'Project Discovery',      'sfp_projectdiscovery',    'api_key',                                          'Freemium', 'Medium', 'Search'),
    (19, 'ZoomEye',                'sfp_zoomeye',             'api_key',                                          'Paid',     'Medium', 'Search'),

    # 3. Blockchain & Cryptocurrency
    (20, 'Advanced Blockchain',    'sfp_blockchain_analytics','blockcypher_api_key,etherscan_api_key',             'Mixed',    'Low',    'Blockchain'),
    (21, 'Arbitrum',               'sfp_arbitrum',            'api_key',                                          'Free',     'Low',    'Blockchain'),
    (22, 'Bitcoin WhoIsWho',       'sfp_bitcoinwhoswho',      'api_key',                                          'Freemium', 'Low',    'Blockchain'),
    (23, 'BNB Chain',              'sfp_bnb',                 'api_key',                                          'Free',     'Low',    'Blockchain'),
    (24, 'Ethereum',               'sfp_ethereum',            'api_key',                                          'Free',     'Low',    'Blockchain'),
    (25, 'Etherscan',              'sfp_etherscan',           'api_key',                                          'Free',     'Medium', 'Blockchain'),
    (26, 'Tron',                   'sfp_tron',                'api_key',                                          'Free',     'Low',    'Blockchain'),

    # 4. Email & Identity
    (27, 'Dehashed',               'sfp_dehashed',            'api_key_username,api_key',                         'Paid',     'High',   'Email'),
    (28, 'EmailCrawlr',            'sfp_emailcrawlr',         'api_key',                                          'Paid',     'Medium', 'Email'),
    (29, 'EmailRep',               'sfp_emailrep',            'api_key',                                          'Free',     'Medium', 'Email'),
    (30, 'FullContact',            'sfp_fullcontact',         'api_key',                                          'Paid',     'Medium', 'Email'),
    (31, 'HaveIBeenPwned',         'sfp_haveibeenpwned',      'api_key',                                          'Free',     'High',   'Email'),
    (32, 'Hunter.io',              'sfp_hunter',              'api_key',                                          'Freemium', 'High',   'Email'),
    (33, 'LeakCheck',              'sfp_leakcheck',           'api_key',                                          'Paid',     'High',   'Email'),
    (34, 'LeakIX',                 'sfp_leakix',              'api_key',                                          'Free',     'Medium', 'Email'),
    (35, 'Snov.io',                'sfp_snov',                'api_key_client_id,api_key_client_secret',           'Freemium', 'Medium', 'Email'),

    # 5. Domain & DNS
    (36, 'CertSpotter',            'sfp_certspotter',         'api_key',                                          'Free',     'High',   'Domain'),
    (37, 'CIRCL.LU',               'sfp_circllu',             'api_key_login,api_key_password',                   'Free',     'Medium', 'Domain'),
    (38, 'Cisco Umbrella',         'sfp_cisco_umbrella',      'api_key',                                          'Paid',     'High',   'Domain'),
    (39, 'DNSGrep',                'sfp_dnsgrep',             '(CSRF auto-managed)',                              'Free',     'Low',    'Domain'),
    (40, 'HostIO',                 'sfp_hostio',              'api_key',                                          'Freemium', 'Medium', 'Domain'),
    (41, 'JsonWHOIS.com',          'sfp_jsonwhoiscom',        'api_key',                                          'Freemium', 'Low',    'Domain'),
    (42, 'SecurityTrails',         'sfp_securitytrails',      'api_key',                                          'Paid',     'High',   'Domain'),
    (43, 'ViewDNS.info',           'sfp_viewdns',             'api_key',                                          'Freemium', 'Medium', 'Domain'),
    (44, 'WhoisFreaks',            'sfp_whoisfreaks',         'api_key',                                          'Freemium', 'Medium', 'Domain'),
    (45, 'Whoisology',             'sfp_whoisology',          'api_key',                                          'Paid',     'Medium', 'Domain'),
    (46, 'Whoxy',                  'sfp_whoxy',               'api_key',                                          'Freemium', 'Low',    'Domain'),
    (47, 'ZoneFile.io',            'sfp_zonefiles',           'api_key',                                          'Paid',     'Low',    'Domain'),
    (48, 'Zetalytics',             'sfp_zetalytics',          'api_key',                                          'Paid',     'Low',    'Domain'),

    # 6. IP & Geolocation
    (49, 'AbstractAPI',            'sfp_abstractapi',         'companyenrichment_api_key,phonevalidation_api_key,ipgeolocation_api_key', 'Freemium', 'Medium', 'IP'),
    (50, 'CriminalIP',             'sfp_criminalip',          'api_key',                                          'Freemium', 'High',   'IP'),
    (51, 'Fraudguard',             'sfp_fraudguard',          'fraudguard_api_key_account,fraudguard_api_key_password', 'Paid', 'Medium', 'IP'),
    (52, 'IPInfo.io',              'sfp_ipinfo',              'api_key',                                          'Freemium', 'High',   'IP'),
    (53, 'IPQualityScore',         'sfp_ipqualityscore',      'api_key',                                          'Freemium', 'Medium', 'IP'),
    (54, 'IP Registry',            'sfp_ipregistry',          'api_key',                                          'Freemium', 'Medium', 'IP'),
    (55, 'IPStack',                'sfp_ipstack',             'api_key',                                          'Freemium', 'Medium', 'IP'),
    (56, 'ipapi.com',              'sfp_ipapicom',            'api_key',                                          'Freemium', 'Medium', 'IP'),
    (57, 'iknowwhatyoudownload',   'sfp_iknowwhatyoudownload','api_key',                                          'Paid',     'Low',    'IP'),
    (58, 'Netlas',                 'sfp_netlas',              'api_key',                                          'Freemium', 'Medium', 'IP'),
    (59, 'NetworksDB',             'sfp_networksdb',          'api_key',                                          'Paid',     'Low',    'IP'),
    (60, 'spur.us',                'sfp_spur',                'api_key',                                          'Paid',     'Low',    'IP'),
    (61, 'UnwiredLabs',            'sfp_unwiredlabs',         'api_key',                                          'Free',     'Low',    'IP'),

    # 7. Social Media & Communication
    (62, 'Bluesky',                'sfp_bluesky',             'access_token',                                     'Free',     'Low',    'Social'),
    (63, 'Discord',                'sfp_discord',             'bot_token',                                        'Free',     'Low',    'Social'),
    (64, 'Instagram',              'sfp_instagram',           'access_token',                                     'Paid',     'Low',    'Social'),
    (65, 'Mastodon',               'sfp_mastodon',            'access_token',                                     'Free',     'Low',    'Social'),
    (66, 'Matrix',                 'sfp_matrix',              'access_token',                                     'Free',     'Low',    'Social'),
    (67, 'Mattermost',             'sfp_mattermost',          'access_token',                                     'Free',     'Low',    'Social'),
    (68, 'Reddit',                 'sfp_reddit',              'client_secret,client_id',                          'Free',     'Medium', 'Social'),
    (69, 'Rocket.Chat',            'sfp_rocketchat',          'access_token',                                     'Free',     'Low',    'Social'),
    (70, 'Social Links',           'sfp_sociallinks',         'api_key',                                          'Paid',     'Low',    'Social'),
    (71, 'Social Profiles',        'sfp_socialprofiles',      'bing_api_key,google_api_key',                      'Paid',     'Medium', 'Social'),
    (72, 'TikTok OSINT',           'sfp_tiktok_osint',        'api_key,api_secret',                               'Free',     'Low',    'Social'),
    (73, 'WeChat',                 'sfp_wechat',              'api_key',                                          'Paid',     'Low',    'Social'),
    (74, 'WhatsApp',               'sfp_whatsapp',            'api_key',                                          'Paid',     'Low',    'Social'),

    # 8. Business Intelligence
    (75, 'BuiltWith',              'sfp_builtwith',           'api_key',                                          'Paid',     'Medium', 'Business'),
    (76, 'c99',                    'sfp_c99',                 'api_key',                                          'Paid',     'Low',    'Business'),
    (77, 'Deepinfo',               'sfp_deepinfo',            'api_key',                                          'Paid',     'Low',    'Business'),
    (78, 'Focsec',                 'sfp_focsec',              'api_key',                                          'Paid',     'Low',    'Business'),
    (79, 'Fofa',                   'sfp_fofa',                'api_key',                                          'Freemium', 'Medium', 'Business'),
    (80, 'FullHunt',               'sfp_fullhunt',            'api_key',                                          'Paid',     'Medium', 'Business'),
    (81, 'Leak-Lookup (Citadel)',   'sfp_citadel',            'api_key',                                          'Paid',     'High',   'Business'),
    (82, 'Luminar',                'sfp_luminar',             'api_key',                                          'Paid',     'Low',    'Business'),
    (83, 'NameAPI',                'sfp_nameapi',             'api_key',                                          'Paid',     'Low',    'Business'),
    (84, 'numverify',              'sfp_numverify',           'api_key',                                          'Freemium', 'Low',    'Business'),
    (85, 'NeutrinoAPI',            'sfp_neutrinoapi',         'api_key',                                          'Freemium', 'Low',    'Business'),
    (86, 'OpenCorporates',         'sfp_opencorporates',      'api_key',                                          'Freemium', 'Low',    'Business'),
    (87, 'Onyphe',                 'sfp_onyphe',              'api_key',                                          'Freemium', 'Medium', 'Business'),
    (88, 'RocketReach',            'sfp_rocketreach',         'api_key',                                          'Paid',     'Medium', 'Business'),
    (89, 'Seon',                   'sfp_seon',                'api_key',                                          'Paid',     'Low',    'Business'),
    (90, 'StackOverflow',          'sfp_stackoverflow',       'api_key',                                          'Free',     'Low',    'Business'),
    (91, 'WhatCMS',                'sfp_whatcms',             'api_key',                                          'Freemium', 'Low',    'Business'),

    # 9. Malware & Security Analysis
    (92, 'Grayhat Warfare',        'sfp_grayhatwarfare',      'api_key',                                          'Paid',     'Low',    'Malware'),
    (93, 'Koodous',                'sfp_koodous',             'api_key',                                          'Free',     'Low',    'Malware'),
    (94, 'MalwarePatrol',          'sfp_malwarepatrol',       'api_key',                                          'Freemium', 'Low',    'Malware'),
    (95, 'MetaDefender',           'sfp_metadefender',        'api_key',                                          'Freemium', 'Low',    'Malware'),
    (96, 'Project Honey Pot',      'sfp_honeypot',            'api_key',                                          'Free',     'Low',    'Malware'),

    # 10. Communication & Specialized
    (97,  'Twilio',                'sfp_twilio',              'api_key_account_sid,api_key_auth_token',            'Paid',     'Low',    'Comms'),
    (98,  'TextMagic',             'sfp_textmagic',           'api_key_username,api_key',                         'Paid',     'Low',    'Comms'),
    (99,  'Onion.link',            'sfp_onioncity',           'api_key',                                          'Paid',     'Low',    'Comms'),
    (100, 'WiGLE',                 'sfp_wigle',               'api_key_encoded',                                  'Free',     'Low',    'Comms'),
    (101, 'API Key Leak Detector', 'sfp_apileak',             'github_token',                                     'Free',     'Low',    'Comms'),
    (102, 'AI Summary',            'sfp_ai_summary',          'api_key',                                          'Paid',     'Low',    'Comms'),
    (103, 'BotScout',              'sfp_botscout',            'api_key',                                          'Free',     'Low',    'Comms'),
    (104, 'F-Secure Riddler.io',   'sfp_fsecure_riddler',     'password',                                         'Paid',     'Low',    'Comms'),
    (105, 'IntelligenceX',         'sfp_intelx',              'api_key',                                          'Paid',     'High',   'Comms'),
    (106, 'PasteBin',              'sfp_pastebin',            'api_key',                                          'Freemium', 'Medium', 'Comms'),
    (107, 'XForce Exchange',       'sfp_xforce',              'xforce_api_key,xforce_api_key_password',           'Paid',     'Medium', 'Comms'),
    (108, 'Tool - Nmap',           'sfp_tool_nmap',           'remote_password',                                  'N/A',      'Medium', 'Tools'),
    (109, 'Tool - Nuclei',         'sfp_tool_nuclei',         'remote_password',                                  'N/A',      'Medium', 'Tools'),
    (110, 'Tool - Gobuster',       'sfp_tool_gobuster',       'remote_password',                                  'N/A',      'Low',    'Tools'),
    (111, 'Tool - PhoneInfoga',    'sfp_tool_phoneinfoga',    'api_key,remote_password',                          'Free',     'Low',    'Tools'),
    (112, 'Tool - Wappalyzer',     'sfp_tool_wappalyzer',     'wappalyzer_api_key',                               'Paid',     'Low',    'Tools'),
    (113, 'Database Storage',      'sfp__stor_db',            'postgresql_password',                              'N/A',      'N/A',    'Storage'),
    (114, 'ElasticSearch Storage', 'sfp__stor_elasticsearch', 'api_key,password',                                 'N/A',      'N/A',    'Storage'),
]

print(f'All-services catalog: {len(ALL_116_SERVICES)} entries')

In [ ]:
# Query the full catalog - filter by tier/priority/category
def show_catalog(tier=None, priority=None, category=None):
    results = ALL_116_SERVICES
    if tier:     results = [r for r in results if r[4] == tier]
    if priority: results = [r for r in results if r[5] == priority]
    if category: results = [r for r in results if r[6] == category]

    print(f"{'#':<4} {'Service':<28} {'Module':<28} {'Tier':<10} {'Priority':<8} {'Cat'}")
    print('-' * 100)
    for r in results:
        print(f"{r[0]:<4} {r[1]:<28} {r[2]:<28} {r[4]:<10} {r[5]:<8} {r[6]}")
    print(f'\n{len(results)} services')


# Show all Free + High priority
show_catalog(tier='Free', priority='High')

In [ ]:
# Full catalog breakdown by category
from collections import Counter
by_cat  = Counter(r[6] for r in ALL_116_SERVICES)
by_tier = Counter(r[4] for r in ALL_116_SERVICES)
by_pri  = Counter(r[5] for r in ALL_116_SERVICES)

print('By category:'); [print(f'  {k}: {v}') for k, v in by_cat.most_common()]
print('\nBy tier:');    [print(f'  {k}: {v}') for k, v in by_tier.most_common()]
print('\nBy priority:');[print(f'  {k}: {v}') for k, v in by_pri.most_common()]

---
## 11. SpiderFoot Configuration Methods

### Method 1: Web UI (Recommended)
1. Navigate to `Settings` → `Module Settings`
2. Find the module and enter the API key
3. Save — keys are encrypted automatically

### Method 2: Environment Variables
```bash
export VIRUSTOTAL_API_KEY="your_key_here"
export SHODAN_API_KEY="your_key_here"
export HUNTER_API_KEY="your_key_here"
export SECURITYTRAILS_API_KEY="your_key_here"
export HIBP_API_KEY="your_key_here"
./sfcli.py
```

### Method 3: .env File
Create `/stuff/spiderfoot/.env`:
```bash
VIRUSTOTAL_API_KEY=your_virustotal_key
SHODAN_API_KEY=your_shodan_key
CENSYS_API_KEY_UID=your_censys_uid
CENSYS_API_KEY_SECRET=your_censys_secret
HUNTER_API_KEY=your_hunter_key
HAVEIBEENPWNED_API_KEY=your_hibp_key
SECURITYTRAILS_API_KEY=your_securitytrails_key
```
Then: `source .env && ./sfcli.py`

### Security Best Practices
- Keys stored in SpiderFoot DB are Fernet-encrypted (`enc:` prefix)
- Use env vars in production — keys never touch disk
- Use unique keys per environment (dev/staging/prod)
- Many free tiers have daily/monthly rate limits — monitor usage

### Verification Checklist
Before running full scans:
- [ ] API key correctly formatted
- [ ] Key still active (not revoked)
- [ ] Rate limits suitable for use case
- [ ] Test endpoint returns valid response
- [ ] Key has required permissions/scopes

---
## 12. SpiderFoot DB Utilities

Query the actual SpiderFoot database to see what's already configured.

In [ ]:
# Check what keys are already set in SpiderFoot's own database
sf_db = Path(SPIDERFOOT_DB)
if sf_db.exists():
    with sqlite3.connect(SPIDERFOOT_DB) as conn:
        try:
            rows = conn.execute(
                "SELECT config_name, config_value FROM tbl_config "
                "WHERE config_name LIKE '%api%' OR config_name LIKE '%key%' OR config_name LIKE '%token%' "
                "ORDER BY config_name"
            ).fetchall()
            print(f'SpiderFoot configured keys ({len(rows)} found):')
            for name, val in rows:
                preview = (val[:20] + '...') if val and len(val) > 20 else (val or '(empty)')
                print(f'  {name:<50} {preview}')
        except Exception as e:
            print(f'Query failed: {e}')
else:
    print(f'SpiderFoot DB not found at {SPIDERFOOT_DB}')
    print('Run SpiderFoot at least once to create the database.')

In [ ]:
# Push acquired keys into SpiderFoot's database directly
# WARNING: Only run this if SpiderFoot is NOT running
sf_db = Path(SPIDERFOOT_DB)
if not sf_db.exists():
    print('SpiderFoot DB not found — start SpiderFoot first')
else:
    completed = db.get_completed()
    if not completed:
        print('No completed keys in hunter DB yet')
    else:
        with sqlite3.connect(SPIDERFOOT_DB) as conn:
            for service, api_key in completed:
                config_key = SPIDERFOOT_KEY_MAP.get(service)
                if config_key:
                    conn.execute(
                        "INSERT OR REPLACE INTO tbl_config (config_name, config_value) VALUES (?, ?)",
                        (config_key, api_key)
                    )
                    print(f'  Set {config_key}')
                else:
                    print(f'  No config key mapping for {service} — set via Web UI')
        print('Done. Restart SpiderFoot to pick up changes.')

In [ ]:
# Backup SpiderFoot config to SQL file
import subprocess
sf_db = Path(SPIDERFOOT_DB)
if sf_db.exists():
    result = subprocess.run(
        ['sqlite3', SPIDERFOOT_DB, '.dump tbl_config'],
        capture_output=True, text=True
    )
    backup_path = Path('spiderfoot_config_backup.sql')
    backup_path.write_text(result.stdout)
    print(f'Backup written to {backup_path} ({len(result.stdout)} bytes)')
else:
    print('SpiderFoot DB not found')

---
## 13. Troubleshooting

| Issue | Cause | Solution |
|-------|-------|----------|
| `Invalid API Key` | Typo or expired key | Verify format and regenerate |
| Rate limit errors | Quota exhausted | Wait for reset or upgrade tier |
| `Authentication failed` | Wrong credentials | Check key AND secret are both set |
| Missing results | Module disabled | Enable in Settings → Module Settings |
| Connection timeout | API service down | Check service status page |

### Debug Mode
```bash
./sfcli.py --debug
```
Logs: `spiderfoot/logs/spiderfoot.log`

### Finding a Module's Config Keys
Open `/stuff/spiderfoot/modules/sfp_<name>.py` and look for the `self.opts` dict:
```python
self.opts = {
    'api_key': {
        'value': '',
        'flags': self.FLAG_CREDENTIAL,
        'description': 'API Key for Service'
    }
}
```

In [ ]:
# Find config key names for any module
import ast

def find_module_config_keys(module_name: str) -> list:
    """Extract config key names from a SpiderFoot module file"""
    module_path = Path(f'/stuff/spiderfoot/modules/{module_name}.py')
    if not module_path.exists():
        return [f'File not found: {module_path}']

    src = module_path.read_text()
    # Regex approach: find keys inside self.opts = {...}
    opts_block = re.search(r'self\.opts\s*=\s*\{(.+?)\}\s*\n\s*self\.optdescs', src, re.DOTALL)
    if not opts_block:
        opts_block = re.search(r'self\.opts\s*=\s*\{(.+?)\}', src, re.DOTALL)
    if opts_block:
        keys = re.findall(r"['\"]([a-z_]+)['\"]\s*:", opts_block.group(1))
        return list(dict.fromkeys(keys))  # deduplicated
    return ['Could not parse opts']


# Example: inspect a module
for mod in ['sfp_virustotal', 'sfp_hunter', 'sfp_shodan', 'sfp_censys']:
    keys = find_module_config_keys(mod)
    print(f'{mod}: {keys}')